In [4]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ['GROQ_API_KEY']= os.getenv('GROQ_API_KEY')

from langchain.chat_models import init_chat_model
llm= init_chat_model(model='openai/gpt-oss-20b', model_provider='groq')
from langchain_core.tools import Tool


In [5]:
# custom tool

def multiply(a:int,b:int)->int:
    """ Multiply a and b
    Args:

    a: first int
    b: second int

    """
    return a*b

def addition(a:int,b:int)->int:
    """Addition a and b
    Args:

    a: first int
    b: second int

    """
    return a+b
    

def divide(a:int,b:int)->int:
    """ Divide a and b
    Args:

    a: first int
    b: second int

    """
    return a/b

tool=[multiply,addition,divide]
llm_with_tool= llm.bind_tools(tool)
    

In [7]:
from langgraph.graph import START,StateGraph,END
from langgraph.prebuilt import tools_condition,ToolNode
from langgraph.graph import MessagesState
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import AIMessage,SystemMessage,HumanMessage

In [9]:
sys_msg= SystemMessage(content='Hey you are math teacher perform only math operation otherwise responce humbly')

In [47]:
def assistant(state: MessagesState):
    return {'messages': [llm_with_tool.invoke([sys_msg] + state['messages'])]}

In [48]:
memory= MemorySaver()

In [49]:
#graph
graph= StateGraph(MessagesState)

graph.add_node('assistant',assistant)

graph.add_node('tools',ToolNode(tool))

graph.add_edge(START,'assistant')
graph.add_conditional_edges('assistant',tools_condition,)
graph.add_edge('tools','assistant')
graph.add_edge('assistant',END)
graph.add_edge('tools',END)


In [50]:
graph=graph.compile(interrupt_before=['assistant'],checkpointer=memory)

In [51]:
thread_id={'configurable':{'thread_id':'123'}}

In [52]:
for event in graph.stream({'messages':HumanMessage(content='  ')},config=thread_id,stream_mode='values'):
    event['messages'][-1].pretty_print()

================================ Human Message =================================

  


In [53]:
state= graph.get_state(thread_id)
state.next

('assistant',)

In [54]:
graph.get_state_history(thread_id)

<generator object Pregel.get_state_history at 0x000002BC553A8720>

In [55]:
for event in graph.stream({'messages':HumanMessage(content='tell me 2+3')},config=thread_id,stream_mode='values'):
    event['messages'][-1].pretty_print()

================================ Human Message =================================

tell me 2+3


In [56]:
for event in graph.stream(None,config=thread_id,stream_mode='values'):
    event['messages'][-1].pretty_print()

================================ Human Message =================================

tell me 2+3
================================== Ai Message ==================================
Tool Calls:
  addition (fc_a817c056-3562-4a2f-8043-71f4cb9eb5f5)
 Call ID: fc_a817c056-3562-4a2f-8043-71f4cb9eb5f5
  Args:
    a: 2
    b: 3
================================= Tool Message =================================
Name: addition

5
